# TextureAnalyser
`capabilities/analysis/texture_analyser.py`

Detects surface texture anomalies like disease spots, powdery mildew, and rust  
using **LBP** (Local Binary Patterns) and **Gabor filters**.  
**No model download. Pure OpenCV. < 50 ms per image.**

| Signal | What it means |
|--------|---------------|
| `lbp_entropy` | Higher = more complex/spotted texture |
| `lbp_uniformity` | Higher = regular structured patterns (spots, mildew) |
| `gabor_energy_mean` | Overall texture energy across all frequencies |
| `anomaly_score` | 0–1 composite score — higher = more unusual texture |
| `texture_class` | `smooth`, `moderate`, or `rough/spotted` |

## Setup

In [ ]:
import sys, os, tempfile
sys.path.insert(0, os.path.abspath('../..'))

import cv2
import numpy as np
import matplotlib.pyplot as plt

from capabilities.analysis.texture_analyser import TextureAnalyser
from sprout_detection.utils.image_gen import make_sprout_image, make_bare_soil_image

analyser = TextureAnalyser()
tmp = tempfile.mkdtemp()
print('TextureAnalyser ready')

## Generate test images — smooth vs diseased

In [ ]:
def make_diseased_image(seed=42, n_spots=40):
    """Create a synthetic 'diseased' leaf image with dark spots."""
    rng = np.random.default_rng(seed)
    img = make_sprout_image(seed=seed)
    h, w = img.shape[:2]
    for _ in range(n_spots):
        cx = int(rng.integers(20, w-20))
        cy = int(rng.integers(20, h-20))
        r  = int(rng.integers(3, 12))
        color = (int(rng.integers(20, 60)), int(rng.integers(20, 60)), int(rng.integers(20, 60)))
        cv2.circle(img, (cx, cy), r, color, -1)
    return img

smooth_path  = os.path.join(tmp, 'smooth.png')
diseased_path = os.path.join(tmp, 'diseased.png')
bare_path    = os.path.join(tmp, 'bare.png')

cv2.imwrite(smooth_path,   make_sprout_image(seed=1))
cv2.imwrite(diseased_path, make_diseased_image(seed=1, n_spots=50))
cv2.imwrite(bare_path,     make_bare_soil_image(seed=1))

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, (p, t) in zip(axes, [(smooth_path,'Smooth (healthy)'), (diseased_path,'Diseased (spots)'), (bare_path,'Bare soil')]):
    ax.imshow(cv2.cvtColor(cv2.imread(p), cv2.COLOR_BGR2RGB))
    ax.set_title(t, fontsize=11); ax.axis('off')
plt.tight_layout(); plt.show()

## Basic usage

In [ ]:
result = analyser.analyse(smooth_path)

print(f'Texture class    : {result.texture_class}')
print(f'Anomaly score    : {result.anomaly_score:.3f}')
print(f'LBP entropy      : {result.lbp_entropy:.3f}  (higher = more complex)')
print(f'LBP uniformity   : {result.lbp_uniformity:.3f}  (higher = regular patterns)')
print(f'Gabor energy mean: {result.gabor_energy_mean:.4f}')
print(f'Gabor energy std : {result.gabor_energy_std:.4f}')
print(f'Inference time   : {result.duration_ms:.1f} ms')

## Compare smooth vs diseased

In [ ]:
cases = {
    'Smooth (healthy)': analyser.analyse(smooth_path),
    'Diseased (spots)': analyser.analyse(diseased_path),
    'Bare soil':        analyser.analyse(bare_path),
}

metrics = ['anomaly_score', 'lbp_entropy', 'lbp_uniformity', 'gabor_energy_mean']
colors  = ['#2ecc71', '#e74c3c', '#95a5a6']

fig, axes = plt.subplots(1, 4, figsize=(15, 4))
for ax, metric in zip(axes, metrics):
    vals   = [getattr(r, metric) for r in cases.values()]
    labels = list(cases.keys())
    bars = ax.bar(labels, vals, color=colors, edgecolor='white')
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x()+bar.get_width()/2, v*1.02, f'{v:.3f}',
                ha='center', va='bottom', fontsize=8)
    ax.set_title(metric.replace('_',' ').title(), fontsize=9)
    ax.tick_params(axis='x', labelsize=8)

plt.suptitle('Texture Metrics — Smooth vs Diseased vs Bare', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## LBP histogram — texture fingerprint

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4), sharey=True)

palette = ['#2ecc71', '#e74c3c', '#95a5a6']
for ax, (name, r), col in zip(axes, cases.items(), palette):
    x = np.arange(len(r.lbp_histogram))
    ax.bar(x, r.lbp_histogram, color=col, alpha=0.8, edgecolor='none')
    ax.set_title(f'{name}\nentropy={r.lbp_entropy:.2f}  uniformity={r.lbp_uniformity:.2f}', fontsize=9)
    ax.set_xlabel('LBP code')

axes[0].set_ylabel('Normalised frequency')
plt.suptitle('LBP Histograms — Texture Fingerprints', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## Gabor filter bank responses — directional texture

In [ ]:
smooth_r   = cases['Smooth (healthy)']
diseased_r = cases['Diseased (spots)']

keys   = list(smooth_r.gabor_responses.keys())
s_vals = [smooth_r.gabor_responses[k]   for k in keys]
d_vals = [diseased_r.gabor_responses[k] for k in keys]

x = np.arange(len(keys))
width = 0.4

fig, ax = plt.subplots(figsize=(14, 4))
ax.bar(x - width/2, s_vals, width, label='Smooth',   color='#2ecc71', alpha=0.85)
ax.bar(x + width/2, d_vals, width, label='Diseased', color='#e74c3c', alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels(keys, rotation=45, ha='right', fontsize=7)
ax.set_ylabel('Energy')
ax.set_title('Gabor Filter Bank Response — Smooth vs Diseased', fontsize=11)
ax.legend()
plt.tight_layout()
plt.show()

## Custom LBP parameters for finer-grained analysis

In [ ]:
# Larger radius captures broader patterns (useful for rust pustules)
analyser_fine = TextureAnalyser(lbp_radius=5, lbp_n_points=40)

r_fine = analyser_fine.analyse(diseased_path)
r_base = cases['Diseased (spots)']

print('Diseased leaf — default vs fine-grain LBP:')
print(f'  Default (r=3): entropy={r_base.lbp_entropy:.3f}  anomaly={r_base.anomaly_score:.3f}')
print(f'  Fine   (r=5): entropy={r_fine.lbp_entropy:.3f}  anomaly={r_fine.anomaly_score:.3f}')

## Using a segmentation mask

In [ ]:
from capabilities.segmentation.hsv_segmentor import HSVSegmentor

seg = HSVSegmentor()
seg_result = seg.segment(diseased_path, profile='green_plant')

result_masked = analyser.analyse(diseased_path, mask=seg_result.mask)
result_full   = analyser.analyse(diseased_path)

print('Diseased leaf — with vs without segmentation mask:')
print(f'  Full image : anomaly={result_full.anomaly_score:.3f}  class={result_full.texture_class}')
print(f'  Plant only : anomaly={result_masked.anomaly_score:.3f}  class={result_masked.texture_class}')
print()
print('Masking to plant pixels only makes the anomaly score more accurate')
print('because background soil texture no longer dilutes the signal.')